In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

In [5]:
results_file = '/Users/tomasandrade/Documents/BSC/ICHOIR/applio/Applio_LS/asv_experiments/scores_multiscale.csv'
df = pd.read_csv(results_file)

In [8]:
horizons = [1, 5, 10, 25, 50]
score_cols = [f'score_h{h}' for h in horizons]

# Ratio score
df['score_ratio'] = df['score_h50'] / df['score_h1'].clip(lower=1e-6)

# Slope score
log_h = np.log(horizons)
df['score_slope'] = df[score_cols].apply(
    lambda row: np.polyfit(log_h, row.values, 1)[0], axis=1
)

y_true = (df['key'] == 'spoof').astype(int)
for col in ['score', 'score_ratio', 'score_slope', 'score_h1', 'score_h50']:
    auc = roc_auc_score(y_true, df[col])
    print(f"{col:20s}  AUC {auc:.4f}  (flipped: {1-auc:.4f})")

score                 AUC 0.4871  (flipped: 0.5129)
score_ratio           AUC 0.5536  (flipped: 0.4464)
score_slope           AUC 0.5482  (flipped: 0.4518)
score_h1              AUC 0.3901  (flipped: 0.6099)
score_h50             AUC 0.5069  (flipped: 0.4931)


In [ ]:
import pandas as pd

In [2]:
df = pd.read_csv('~/Desktop/silence_diagnostics/flagged_silences.csv')

# Count how many flags each file has
flag_cols = ['flag_long_internal', 'flag_high_fraction', 
             'flag_short_speech', 'flag_many_segments']
df['n_flags'] = df[flag_cols].sum(axis=1)

# Worst offenders — multiple flags AND long internal silence
priority = df[df['n_flags'] >= 2].sort_values(
    'max_internal_sil_s', ascending=False
)
print(f"Files with 2+ flags: {len(priority)}")
print(priority[['name', 'duration_s', 'max_internal_sil_s', 
                'silence_fraction', 'n_flags']].head(30))

Files with 2+ flags: 422
            name  duration_s  max_internal_sil_s  silence_fraction  n_flags
0   LA_T_4982302       6.400                3.98             0.656        2
1   LA_T_9215136       6.440                3.92             0.630        2
2   LA_T_1129020       8.500                3.40             0.786        2
3   LA_T_9690282       4.620                3.26             0.753        2
4   LA_T_3448136       4.580                3.22             0.729        2
5   LA_T_7521161       8.880                2.80             0.869        2
6   LA_T_6960204       4.900                1.98             0.547        2
7   LA_T_8503056       4.400                1.94             0.700        2
8   LA_T_5480079       2.940                1.94             0.741        2
9   LA_T_6010560       4.460                1.68             0.628        2
10  LA_T_5706600       4.420                1.64             0.665        2
11  LA_T_8719032       4.420                1.64             0.

In [6]:
cm_file = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASVspoof2019/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
protocol = pd.read_csv(
    cm_file,
    sep=' ', header=None,
    names=['speaker_id', 'file_name', 'placeholder', 'system_id', 'key']
)

# Merge on filename (flagged 'name' matches protocol 'file_name')
flagged = df.merge(
    protocol[['file_name', 'key', 'system_id']],
    left_on='name', right_on='file_name',
    how='left'
)

In [7]:
print(flagged['key'].value_counts())
print(flagged['key'].value_counts(normalize=True))

key
spoof       2312
bonafide     100
Name: count, dtype: int64
key
spoof       0.958541
bonafide    0.041459
Name: proportion, dtype: float64


In [8]:
print(flagged[flagged['key']=='spoof']['system_id'].value_counts())
print(flagged[flagged['key']=='spoof']['system_id'].value_counts(normalize=True))

system_id
A04    772
A01    519
A03    333
A02    331
A05    214
A06    143
Name: count, dtype: int64
system_id
A04    0.333910
A01    0.224481
A03    0.144031
A02    0.143166
A05    0.092561
A06    0.061851
Name: proportion, dtype: float64


In [1]:
import pandas as pd

In [2]:
results_transf = '/Users/tomasandrade/Documents/BSC/ICHOIR/applio/Applio_LS/asv_experiments/scores_transformer_official.csv'

In [33]:
scores_df = pd.read_csv(results_transf)

# Find the bonafide outliers — what are they?
bf_outliers = scores_df[
    (scores_df['key'] == 'bonafide') & 
    (scores_df['score'] > scores_df[scores_df['key']=='bonafide']['score'].quantile(0.95))
]

bf_outliers_top20 = bf_outliers[['name', 'score']].sort_values('score', ascending=False).head(20)
print(bf_outliers_top20)

               name     score
12476  LA_D_4574274  0.465760
23093  LA_D_3559270  0.461496
19320  LA_D_7152817  0.456548
14769  LA_D_3514242  0.454183
9799   LA_D_8718356  0.453032
1159   LA_D_6292198  0.449890
8984   LA_D_8220866  0.449774
11562  LA_D_4021153  0.445016
14811  LA_D_9685348  0.444156
12792  LA_D_6032630  0.440602
14344  LA_D_5495040  0.438160
7663   LA_D_6027798  0.435829
5686   LA_D_4777626  0.435683
13743  LA_D_3579781  0.434247
7028   LA_D_4969607  0.433190
6086   LA_D_7618637  0.432675
8679   LA_D_6622348  0.432355
12013  LA_D_6252124  0.432056
17460  LA_D_3174998  0.431609
13259  LA_D_2339825  0.430927


In [5]:
# Per-system outlier rates
for sys in scores_df['system_id'].unique():
    subset = scores_df[scores_df['system_id'] == sys]['score']
    q99 = scores_df['score'].quantile(0.99)
    outlier_rate = (subset > q99).mean()
    print(f"{sys}: {outlier_rate:.1%} above 99th percentile")

A01: 1.2% above 99th percentile
A06: 0.0% above 99th percentile
A05: 0.0% above 99th percentile
A03: 1.5% above 99th percentile
-: 0.5% above 99th percentile
A02: 1.2% above 99th percentile
A04: 2.4% above 99th percentile


In [14]:
bf_outliers_top20['set'] = bf_outliers_top20['name'].str.split('_', expand=True)[1]

In [17]:
bf_outliers_top20['file'] = bf_outliers_top20['name'] + '.flac'

In [30]:
LA_ROOT = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASVspoof2019/LA'
dev_flac = f'{LA_ROOT}/ASVspoof2019_LA_dev/flac/'
train_flac = f'{LA_ROOT}/ASVspoof2019_LA_train/flac/'

In [31]:
bf_outliers_top20['path'] = dev_flac + bf_outliers_top20['file'] 

In [27]:
out_dir = '/Users/tomasandrade/Desktop/test_flac'

In [28]:
import shutil

In [32]:
for idx, row in bf_outliers_top20.iterrows():
    shutil.copy2(row['path'], out_dir) 

In [22]:
bf_outliers_top20.sort_values('name')

,name,score,set,file
13259,LA_D_2339825,0.430927,D,LA_D_2339825.flac
17460,LA_D_3174998,0.431609,D,LA_D_3174998.flac
14769,LA_D_3514242,0.454183,D,LA_D_3514242.flac
23093,LA_D_3559270,0.461496,D,LA_D_3559270.flac
13743,LA_D_3579781,0.434247,D,LA_D_3579781.flac
11562,LA_D_4021153,0.445016,D,LA_D_4021153.flac
12476,LA_D_4574274,0.465760,D,LA_D_4574274.flac
5686,LA_D_4777626,0.435683,D,LA_D_4777626.flac
7028,LA_D_4969607,0.433190,D,LA_D_4969607.flac
14344,LA_D_5495040,0.438160,D,LA_D_5495040.flac


In [39]:
# Check speaker ID of bonafide outliers
bf_outlier_names = scores_df[
    (scores_df['key'] == 'bonafide') &
    (scores_df['score'] > scores_df[scores_df['key']=='bonafide']['score'].quantile(0.95))
]['name'].tolist()


# Cross-reference with protocol file
root_LA = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASVspoof2019/LA/'#
cm_dev_file = f'{root_LA}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'

protocol = pd.read_csv(cm_dev_file,
    sep=' ', header=None,
    names=['speaker_id', 'file_name', 'placeholder', 'system_id', 'key'])

outlier_speakers = protocol[protocol['file_name'].isin(bf_outlier_names)]['speaker_id'].value_counts()
print(outlier_speakers)

speaker_id
LA_0073    89
LA_0070     9
LA_0078     7
LA_0102     5
LA_0069     4
LA_0104     3
LA_0076     2
LA_0077     2
LA_0099     2
LA_0106     2
LA_0071     1
LA_0103     1
LA_0108     1
Name: count, dtype: int64


In [40]:
# AUC with and without LA_0073
from sklearn.metrics import roc_auc_score

scores_df = pd.read_csv(results_transf)

# Full AUC
y_true  = (scores_df['key'] == 'spoof').astype(int)
auc_full = roc_auc_score(y_true, scores_df['score'])
print(f"AUC full dataset    : {auc_full:.4f}")

# AUC excluding LA_0073
# Need speaker_id in scores_df — merge with protocol first
protocol = pd.read_csv(cm_dev_file,
    sep=' ', header=None,
    names=['speaker_id', 'file_name', 'placeholder', 'system_id', 'key'])

scores_with_spk = scores_df.merge(
    protocol[['file_name', 'speaker_id']],
    left_on='name', right_on='file_name', how='left'
)

mask_no73 = scores_with_spk['speaker_id'] != 'LA_0073'
subset    = scores_with_spk[mask_no73]
y_true_no73  = (subset['key'] == 'spoof').astype(int)
auc_no73     = roc_auc_score(y_true_no73, subset['score'])
print(f"AUC excluding LA_0073: {auc_no73:.4f}")

# How many bonafide utterances is LA_0073?
n_0073 = (scores_with_spk['speaker_id'] == 'LA_0073').sum()
n_total_bf = (scores_df['key'] == 'bonafide').sum()
print(f"\nLA_0073 utterances: {n_0073} ({n_0073/n_total_bf*100:.1f}% of bonafide)")

# Score distribution for LA_0073 vs other bonafide
spk73_scores  = scores_with_spk[scores_with_spk['speaker_id']=='LA_0073']['score']
other_bf      = scores_with_spk[
    (scores_with_spk['key']=='bonafide') & 
    (scores_with_spk['speaker_id']!='LA_0073')
]['score']
print(f"\nLA_0073 mean score    : {spk73_scores.mean():.4f} ± {spk73_scores.std():.4f}")
print(f"Other bonafide mean   : {other_bf.mean():.4f} ± {other_bf.std():.4f}")

AUC full dataset    : 0.6530
AUC excluding LA_0073: 0.6680

LA_0073 utterances: 1988 (78.0% of bonafide)

LA_0073 mean score    : 0.3543 ± 0.0709
Other bonafide mean   : 0.3061 ± 0.0254


In [41]:
root_LA = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASVspoof2019/LA/'#
cm_train_file = f'{root_LA}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'

train_protocol = pd.read_csv(cm_train_file,
    sep=' ', header=None,
    names=['speaker_id', 'file_name', 'placeholder', 'system_id', 'key'])

train_bonafide_speakers = train_protocol[
    train_protocol['key'] == 'bonafide'
]['speaker_id'].value_counts()
print(train_bonafide_speakers)
print(f"\nIs LA_0073 in training? {'LA_0073' in train_bonafide_speakers.index}")

speaker_id
LA_0089    132
LA_0082    132
LA_0083    132
LA_0096    132
LA_0095    132
LA_0094    132
LA_0093    132
LA_0092    132
LA_0090    127
LA_0097    127
LA_0091    127
LA_0079    127
LA_0080    127
LA_0088    127
LA_0087    127
LA_0086    127
LA_0085    127
LA_0084    127
LA_0081    127
LA_0098    127
Name: count, dtype: int64

Is LA_0073 in training? False
